## 1. Konfigurasi run

In [ ]:
SEEDS = [64, 65, 66]
EPOCHS = 50
FT_EPOCHS = 20
LIMIT = 0
IMG = 224
CORE = [
    "mobilenetv2",
    "mnv3s",
    "mnv3s_mlp",
    "mnv3s_fuse",
    "mnv3l",
    "mnv3l_mlp",
    "mnv3l_fuse",
]
EXT = ["mnv3s_fmlp", "mnv3l_fmlp"]
RUNS = CORE + EXT

CLAHE_MODELS = ["mnv3s", "mnv3s_fmlp", "mnv3l", "mnv3l_fmlp"]
CLAHE_CLIP, CLAHE_GRID = 2.0, 8
assert all(m in RUNS for m in CLAHE_MODELS)


## 2. Lingkungan dan dataset

In [ ]:
import os, sys, subprocess

ON_KAGGLE = os.path.exists("/kaggle/input")
OUT_DIR = "/kaggle/working/runs" if ON_KAGGLE else "/content/runs"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs("parts", exist_ok=True)

if ON_KAGGLE:
    DATA_DIR = "/kaggle/input"
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
    import kagglehub

    DATA_DIR = kagglehub.dataset_download("elmadafri/the-wildfire-dataset")


## 3. Konfigurasi dan logging (`01_config`)

In [ ]:
%%writefile parts/01_config.py
import os, sys, time, argparse, platform, shutil, hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2
from tqdm.auto import tqdm
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix,
    f1_score,
)

ap = argparse.ArgumentParser()
ap.add_argument("--data", default="/content")
ap.add_argument("--out", default="/content/runs")
ap.add_argument("--cache", default="/tmp/firedet_cache")
ap.add_argument("--seed", type=int, default=64)
ap.add_argument("--model", default="mnv3s")
ap.add_argument("--img", type=int, default=224)
ap.add_argument("--batch", type=int, default=32)
ap.add_argument("--epochs", type=int, default=50)
ap.add_argument("--ft_epochs", type=int, default=20)
ap.add_argument("--lr", type=float, default=1e-3)
ap.add_argument("--ft_lr", type=float, default=1e-5)
ap.add_argument("--unfreeze", type=int, default=30)
ap.add_argument("--dropout", type=float, default=0.3)
ap.add_argument("--fuse_dim", type=int, default=64)
ap.add_argument("--mlp_hidden", type=int, default=64)
ap.add_argument("--target_recall", type=float, default=0.95)
ap.add_argument("--weights", default="imagenet")
ap.add_argument("--limit", type=int, default=0)
ap.add_argument("--clahe", type=int, default=0)
ap.add_argument("--clahe_clip", type=float, default=2.0)
ap.add_argument("--clahe_grid", type=int, default=8)
A = ap.parse_args()

TAG = A.model + ("_clahe" if A.clahe else "")
RUN = Path(A.out) / f"{TAG}_img{A.img}_seed{A.seed}"
if RUN.exists():
    shutil.rmtree(RUN)
RUN.mkdir(parents=True)
T0, S = time.time(), A.img


def log(msg):
    line = f"[{time.strftime('%H:%M:%S')} +{time.time() - T0:5.0f}s] {msg}"
    print(line, flush=True)
    with open(RUN / "run.log", "a") as f:
        f.write(line + "\n")


def append_csv(name, df):
    p = RUN / name
    df.to_csv(p, mode="a", header=not p.exists(), index=False)

## 4. Split data (`02_split`)

In [ ]:
%%writefile parts/02_split.py
SPLITS, CLASSES, EXT = (
    ("train", "val", "test"),
    ("nofire", "fire"),
    (".jpg", ".jpeg", ".png", ".bmp", ".webp"),
)


def find_root(base):
    for dp, dn, _ in os.walk(base):
        if all(s in dn for s in SPLITS) and all(
            {d for d in os.listdir(f"{dp}/{s}") if os.path.isdir(f"{dp}/{s}/{d}")}
            == set(CLASSES)
            for s in SPLITS
        ):
            return dp
    sys.exit(f"Struktur train/val/test/{{fire,nofire}} tidak ditemukan di {base}")


def list_items(split):
    items = [
        (f"{ROOT}/{split}/{c}/{f}", int(c == "fire"))
        for c in CLASSES
        for f in sorted(os.listdir(f"{ROOT}/{split}/{c}"))
        if f.lower().endswith(EXT)
    ]
    if A.limit:
        items = [
            it for k in (0, 1) for it in [i for i in items if i[1] == k][: A.limit]
        ]
    return items


log(
    f"Start | model={TAG} | clahe={A.clahe} | img={S} | seed={A.seed} | epochs={A.epochs}+{A.ft_epochs} | cpu={os.cpu_count()}"
)
ROOT = find_root(A.data)

## 5. Preprocessing (`03_preprocess`)

In [ ]:
%%writefile parts/03_preprocess.py
CTAG = f"_clahe{A.clahe_clip:g}x{A.clahe_grid}" if A.clahe else ""


def prep(path):
    im = cv2.imread(path)
    if im is None:
        return None
    im = cv2.resize(im, (S, S), interpolation=cv2.INTER_AREA)
    if A.clahe:
        l, a, b = cv2.split(cv2.cvtColor(im, cv2.COLOR_BGR2LAB))
        l = cv2.createCLAHE(
            clipLimit=A.clahe_clip, tileGridSize=(A.clahe_grid, A.clahe_grid)
        ).apply(l)
        im = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)


def load(split):
    cf = (
        Path(A.cache)
        / f"{hashlib.md5(ROOT.encode()).hexdigest()[:6]}_img{S}_lim{A.limit}{CTAG}_{split}.npz"
    )
    if cf.exists():
        z = np.load(cf)
        return z["X"], z["y"], z["rel"].tolist(), int(z["skipped"])
    items = list_items(split)
    with ThreadPoolExecutor(os.cpu_count() or 2) as ex:
        imgs = list(
            tqdm(
                ex.map(prep, [p for p, _ in items]),
                total=len(items),
                desc=f"preprocess {split}",
                mininterval=3,
            )
        )
    ok = [i for i, m in enumerate(imgs) if m is not None]
    X, y, rel = (
        np.stack([imgs[i] for i in ok]),
        np.array([items[i][1] for i in ok]),
        [os.path.relpath(items[i][0], ROOT) for i in ok],
    )
    cf.parent.mkdir(parents=True, exist_ok=True)
    np.savez(cf, X=X, y=y, rel=np.array(rel), skipped=len(items) - len(ok))
    return X, y, rel, len(items) - len(ok)


D, skipped = {}, 0
for s in SPLITS:
    X, y, rel, sk = load(s)
    D[s] = (X, y, rel)
    skipped += sk
log(
    "Split (train/val/test): "
    + "/".join(str(len(D[s][1])) for s in SPLITS)
    + f" | gambar korup dilewati: {skipped}"
)

## 6. Seed dan TensorFlow (`04_seed_tf`)

In [ ]:
%%writefile parts/04_seed_tf.py
import tensorflow as tf
from tensorflow.keras import layers, callbacks

tf.keras.utils.set_random_seed(A.seed)
gpus = tf.config.list_physical_devices("GPU")


## 7. Augmentasi (`05_augment`)

In [ ]:
%%writefile parts/05_augment.py


def make_augmentation():
    return tf.keras.Sequential(
        [
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(20 / 360, fill_mode="reflect"),
            layers.RandomTranslation(0.1, 0.1, fill_mode="reflect"),
            layers.RandomBrightness(0.2, value_range=(0, 255)),
        ],
        name="augmentation",
    )

## 8. Model (`06_model`)

In [ ]:
%%writefile parts/06_model.py
MODELS = {"mobilenetv2": ("MobileNetV2", "tf", "gap")}
for _k, _cls in (("mnv3s", "MobileNetV3Small"), ("mnv3l", "MobileNetV3Large")):
    for _suf, _head in (
        ("", "gap"),
        ("_mlp", "mlp"),
        ("_fuse", "fuse"),
        ("_fmlp", "fmlp"),
        ("_fuse3", "fuse3"),
    ):
        MODELS[_k + _suf] = (_cls, "builtin", _head)
assert A.model in MODELS, f"model tidak dikenal: {A.model} (pilih: {list(MODELS)})"
HEAD = MODELS[A.model][2]


def normalize(x):
    mode = MODELS[A.model][1]
    if mode == "tf":
        return layers.Rescaling(1 / 127.5, offset=-1)(x)
    return x


def cba(x, d, k, name):
    x = layers.Conv2D(d, k, padding="same", use_bias=False, name=name + "_conv")(x)
    return layers.Activation("hard_swish", name=name + "_act")(
        layers.BatchNormalization(name=name + "_bn")(x)
    )


def stride_tap(base, stride):
    cand = [
        l
        for l in base.layers
        if isinstance(l, layers.Add) and l.output.shape[1] == S // stride
    ]
    assert cand, f"tap stride-{stride} tidak ditemukan"
    return cand[-1].output


def top_down(fine, coarse, name):
    assert fine.shape[1] == 2 * coarse.shape[1], (
        f"resolusi tidak kompatibel: {fine.shape} vs {coarse.shape}"
    )
    pf, pc = (
        cba(fine, A.fuse_dim, 1, name + "_pf"),
        cba(coarse, A.fuse_dim, 1, name + "_pc"),
    )
    f = layers.Add(name=name + "_add")(
        [pf, layers.UpSampling2D(2, interpolation="bilinear", name=name + "_up")(pc)]
    )
    return cba(
        layers.DepthwiseConv2D(3, padding="same", use_bias=False, name=name + "_dw")(f),
        A.fuse_dim,
        1,
        name + "_smooth",
    )


def make_head(head, c32, c16=None, c8=None):
    if head in ("fuse", "fmlp", "fuse3"):
        f = top_down(c16, c32, "fuse16")
        if head == "fuse3":
            f = top_down(c8, f, "fuse8")
    else:
        f = c32
    p = layers.Dropout(A.dropout)(layers.GlobalAveragePooling2D()(f))
    if head in ("mlp", "fmlp"):
        p = layers.Dense(A.mlp_hidden, activation="hard_swish", name="mlp_hidden")(p)
    return layers.Dense(1, activation="sigmoid", name="p_fire")(p)


def build():
    base = getattr(tf.keras.applications, MODELS[A.model][0])(
        include_top=False,
        input_shape=(S, S, 3),
        weights=None if A.weights == "none" else "imagenet",
    )
    base.trainable = False
    inp = layers.Input((S, S, 3))
    x = normalize(make_augmentation()(inp))
    c8 = c16 = None
    if HEAD == "fuse3":
        c8, c16, c32 = tf.keras.Model(
            base.input,
            [stride_tap(base, 8), stride_tap(base, 16), base.output],
            name="backbone_multiscale",
        )(x, training=False)
    elif HEAD in ("fuse", "fmlp"):
        c16, c32 = tf.keras.Model(
            base.input, [stride_tap(base, 16), base.output], name="backbone_multiscale"
        )(x, training=False)
    else:
        c32 = base(x, training=False)
    return tf.keras.Model(inp, make_head(HEAD, c32, c16, c8)), base

## 9. Kompleksitas (`07_complexity`)

In [ ]:
%%writefile parts/07_complexity.py
from tensorflow.python.framework.convert_to_constants import (
    convert_variables_to_constants_v2,
)


def n_trainable(model):
    return int(sum(np.prod(w.shape) for w in model.trainable_weights))


def analytic_macs(model):
    tot = 0

    def walk(m):
        nonlocal tot
        for l in m.layers:
            if isinstance(l, tf.keras.Model):
                walk(l)
                continue
            try:
                o = l.output.shape
            except Exception:
                continue
            sp = int(np.prod([d for d in o[1:-1] if d])) if len(o) > 2 else 1
            if isinstance(l, layers.DepthwiseConv2D):
                k = l.kernel_size
                tot += sp * k[0] * k[1] * l.input.shape[-1] * l.depth_multiplier
            elif isinstance(l, layers.Conv2D):
                k = l.kernel_size
                tot += sp * k[0] * k[1] * (l.input.shape[-1] // l.groups) * l.filters
            elif isinstance(l, layers.Dense):
                tot += sp * l.input.shape[-1] * l.units

    walk(model)
    return tot


def measure_complexity(model, base):
    fn = tf.function(lambda x: model(x, training=False))
    frozen = convert_variables_to_constants_v2(
        fn.get_concrete_function(tf.TensorSpec([1, S, S, 3], tf.float32))
    )
    with frozen.graph.as_default():
        opt = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
        opt["output"] = "none"
        flops = tf.compat.v1.profiler.profile(
            graph=frozen.graph,
            run_meta=tf.compat.v1.RunMetadata(),
            cmd="scope",
            options=opt,
        ).total_float_ops

    def bench(bs, warm, n):
        x = tf.random.uniform([bs, S, S, 3], 0, 255)
        for _ in range(warm):
            fn(x).numpy()
        t = time.perf_counter()
        for _ in range(n):
            fn(x).numpy()
        return (time.perf_counter() - t) / n

    mac = analytic_macs(model)
    return dict(
        params_backbone=int(base.count_params()),
        params_total=int(model.count_params()),
        params_head=int(model.count_params() - base.count_params()),
        flops_g=2 * mac / 1e9,
        macs_g=mac / 1e9,
        flops_profiler_g=flops / 1e9,
        latency_ms_b1=bench(1, 20, 100) * 1000,
        throughput_ips=A.batch / bench(A.batch, 3, 20),
    )

## 10. Konfigurasi training (`08_training`)

In [ ]:
%%writefile parts/08_training.py


def make_ds(X, y=None, train=False):
    d = tf.data.Dataset.from_tensor_slices(
        X if y is None else (X, y.astype("float32")[:, None])
    )
    if train:
        d = d.shuffle(len(X), seed=A.seed, reshuffle_each_iteration=True)
    f = (
        (lambda x: tf.cast(x, tf.float32))
        if y is None
        else (lambda x, t: (tf.cast(x, tf.float32), t))
    )
    return (
        d.batch(A.batch)
        .map(f, num_parallel_calls=tf.data.AUTOTUNE)
        .prefetch(tf.data.AUTOTUNE)
    )


def compile_(m, lr):
    m.compile(
        tf.keras.optimizers.Adam(lr),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.AUC(name="auc"),
        ],
    )


def class_weights(y):
    return {0: len(y) / (2 * (y == 0).sum()), 1: len(y) / (2 * (y == 1).sum())}


def set_finetune(base):
    base.trainable = True
    for l in base.layers[: -A.unfreeze]:
        l.trainable = False
    for l in base.layers:
        if isinstance(l, layers.BatchNormalization):
            l.trainable = False


class EpochLog(callbacks.Callback):
    def __init__(self, exp, phase):
        super().__init__()
        self.exp, self.phase = exp, phase

    def on_epoch_begin(self, epoch, logs=None):
        self.t0 = time.time()

    def on_epoch_end(self, epoch, logs=None):
        L = logs or {}
        v = self.model.optimizer.learning_rate
        try:
            lr = float(v.numpy())
        except Exception:
            lr = float(np.array(v))
        append_csv(
            "epoch_log.csv",
            pd.DataFrame(
                [
                    dict(
                        exp=self.exp,
                        model=TAG,
                        img=S,
                        seed=A.seed,
                        phase=self.phase,
                        epoch=epoch + 1,
                        loss=L.get("loss"),
                        accuracy=L.get("accuracy"),
                        auc=L.get("auc"),
                        val_loss=L.get("val_loss"),
                        val_accuracy=L.get("val_accuracy"),
                        val_auc=L.get("val_auc"),
                        lr=lr,
                        epoch_seconds=time.time() - self.t0,
                        timestamp=time.strftime("%Y-%m-%d %H:%M:%S"),
                    )
                ]
            ),
        )


def fit(exp, phase, n, lr):
    compile_(model, lr)
    ck = str(RUN / f"{exp.lower()}_best.weights.h5")
    h = model.fit(
        tr,
        validation_data=va,
        epochs=n,
        class_weight=cw,
        verbose=0,
        callbacks=[
            EpochLog(exp, phase),
            callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
            callbacks.ModelCheckpoint(
                ck, monitor="val_loss", save_best_only=True, save_weights_only=True
            ),
        ],
    )
    model.load_weights(ck)
    return h


## 11. Evaluasi (`09_evaluation`)

In [ ]:
%%writefile parts/09_evaluation.py


def metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(
        thr=float(thr),
        acc=(tp + tn) / len(y),
        recall_fire=tp / max(tp + fn, 1),
        precision_fire=tp / max(tp + fp, 1),
        f1_fire=f1_score(y, pred, zero_division=0),
        far=fp / max(fp + tn, 1),
        recall_nofire=tn / max(tn + fp, 1),
        roc_auc=roc_auc_score(y, p),
        pr_auc=average_precision_score(y, p),
        tp=int(tp),
        fp=int(fp),
        tn=int(tn),
        fn=int(fn),
    )


def pick_thr(y, p, target):
    pr, rc, th = precision_recall_curve(y, p)
    pr, rc = pr[:-1], rc[:-1]
    ok = np.where(rc >= target)[0]
    return (
        float(th[ok[np.argmax(pr[ok])]])
        if len(ok)
        else float(th[np.argmax(2 * pr * rc / (pr + rc + 1e-9))])
    )


def evaluate(exp, model, hist, t_start):
    pv = model.predict(make_ds(D["val"][0]), verbose=0).ravel()
    pt = model.predict(make_ds(D["test"][0]), verbose=0).ravel()
    thr = pick_thr(D["val"][1], pv, A.target_recall)
    best = int(np.argmin(hist.history["val_loss"]) + 1)
    try:
        peak = tf.config.experimental.get_memory_info("GPU:0")["peak"] / 2**20
    except Exception:
        peak = None
    cx = dict(
        model=TAG,
        head=HEAD,
        params_head=CX["params_head"],
        params_total=CX["params_total"],
        params_trainable=n_trainable(model),
        flops_g=CX["flops_g"],
        latency_ms_b1=CX["latency_ms_b1"],
        throughput_ips=CX["throughput_ips"],
    )
    rows = [
        dict(
            exp=exp,
            img=S,
            seed=A.seed,
            split=sp,
            thr_type=tt,
            **metrics(y, p, t),
            best_epoch=best,
            seconds=time.time() - t_start,
            peak_mem_mb=peak,
            **cx,
        )
        for sp, y, p in (("val", D["val"][1], pv), ("test", D["test"][1], pt))
        for tt, t in (("0.5", 0.5), ("val", thr))
    ]
    append_csv("results.csv", pd.DataFrame(rows))
    append_csv(
        "preds.csv",
        pd.concat(
            [
                pd.DataFrame(
                    dict(
                        exp=exp,
                        model=TAG,
                        img=S,
                        seed=A.seed,
                        split=sp,
                        path=D[sp][2],
                        y_true=D[sp][1],
                        p_fire=p,
                    )
                )
                for sp, p in (("val", pv), ("test", pt))
            ]
        ),
    )
    r = [x for x in rows if x["split"] == "test" and x["thr_type"] == "val"][0]
    log(
        f"{TAG}-{S}-s{A.seed} {exp} test | thr {thr:.3f}: recall {r['recall_fire']:.4f} "
        f"FAR {r['far']:.4f} | ROC-AUC {r['roc_auc']:.4f} PR-AUC {r['pr_auc']:.4f}"
    )


## 12. Alur eksperimen (`10_run`)

In [ ]:
%%writefile parts/10_run.py
Xtr, ytr, _ = D["train"]
Xva, yva, _ = D["val"]
cw = class_weights(ytr)
tr, va = make_ds(Xtr, ytr, True), make_ds(Xva, yva)
model, base = build()
CX = measure_complexity(model, base)


def write_config(end=None):
    pd.DataFrame(
        [
            dict(
                seed=A.seed,
                **{
                    k: v
                    for k, v in vars(A).items()
                    if k not in ("seed", "model", "clahe")
                },
                optimizer="adam",
                loss="binary_crossentropy",
                class_weight="balanced",
                cw0=cw[0],
                cw1=cw[1],
                head=HEAD,
                early_stopping=False,
                reduce_lr="factor0.5_patience3",
                checkpoint="best_val_loss",
                aug="flip,rot20,shift0.1,bright0.2,reflect",
                clahe=bool(A.clahe),
                arch=A.model,
                model=TAG,
                norm=MODELS[A.model][1],
                precision="float32",
                n_train=len(D["train"][1]),
                n_val=len(D["val"][1]),
                n_test=len(D["test"][1]),
                skipped=skipped,
                **CX,
                tf=tf.__version__,
                python=platform.python_version(),
                cv2=cv2.__version__,
                numpy=np.__version__,
                device=(
                    tf.config.experimental.get_device_details(gpus[0]).get(
                        "device_name", "GPU"
                    )
                    if gpus
                    else "CPU"
                ),
                cpu_count=os.cpu_count(),
                start=time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(T0)),
                end=end,
            )
        ]
    ).to_csv(RUN / "config.csv", index=False)


write_config()

t1 = time.time()
evaluate("Exp1", model, fit("Exp1", "frozen", A.epochs, A.lr), t1)

t2 = time.time()
set_finetune(base)
evaluate("Exp2", model, fit("Exp2", "finetune", A.ft_epochs, A.ft_lr), t2)

write_config(end=time.strftime("%Y-%m-%d %H:%M:%S"))
(RUN / "DONE").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))
log(f"selesai | {(time.time() - T0) / 60:.1f} menit")


## 13. Merge `train.py`

In [ ]:
import glob, py_compile

files = sorted(glob.glob("parts/*.py"))
open("train.py", "w").write("\n\n".join(open(f).read() for f in files))
py_compile.compile("train.py", doraise=True)


## 14. Jalankan semua run

In [ ]:
JOBS = [(m, 0) for m in RUNS] + [(m, 1) for m in CLAHE_MODELS]
for seed in SEEDS:
    for m, cl in JOBS:
        tag = m + ("_clahe" if cl else "")
        if os.path.exists(f"{OUT_DIR}/{tag}_img{IMG}_seed{seed}/DONE"):
            continue
        get_ipython().system(
            f'python -u train.py --data "{DATA_DIR}" --out {OUT_DIR} --seed {seed} --epochs {EPOCHS} --ft_epochs {FT_EPOCHS} --limit {LIMIT} --model {m} --img {IMG} --clahe {cl} --clahe_clip {CLAHE_CLIP} --clahe_grid {CLAHE_GRID} 2>&1 | tee -a {OUT_DIR}/console.log'
        )


## 15. Analisis: tabel metrik (mean ± sd antar seed)

In [ ]:
import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
)

os.makedirs(f"{OUT_DIR}/reports", exist_ok=True)
done = [
    d
    for d in sorted(glob.glob(f"{OUT_DIR}/*_img{IMG}_seed*"))
    if os.path.exists(f"{d}/DONE")
]


def load(f):
    return pd.concat([pd.read_csv(f"{d}/{f}.csv") for d in done], ignore_index=True)


cfg, ep, res, pr = (load(f) for f in ("config", "epoch_log", "results", "preds"))
res["thr_type"] = res.thr_type.astype(str)
models = [m for m in RUNS if m in set(res.model)]
common = sorted(set.intersection(*[set(res[res.model == m].seed) for m in models]))
cfg, ep, res, pr = (
    d[d.seed.isin(common) & d.model.isin(models)].reset_index(drop=True)
    for d in (cfg, ep, res, pr)
)


def far_at_recall(y, p, r):
    fpr, tpr, _ = roc_curve(y, p)
    return float(fpr[np.argmax(tpr >= r)])


rows = []
for (m, s, e), g in pr[pr.split == "test"].groupby(["model", "seed", "exp"]):
    y, p = g.y_true.values, g.p_fire.values
    rows.append(
        dict(
            model=m,
            seed=s,
            exp=e,
            roc_auc=roc_auc_score(y, p),
            pr_auc=average_precision_score(y, p),
            far_r95=far_at_recall(y, p, 0.95),
            far_r90=far_at_recall(y, p, 0.90),
        )
    )
M = pd.DataFrame(rows).set_index(["model", "seed", "exp"])
t = res[res.split == "test"]
key = ["model", "seed", "exp"]
M = M.join(
    t[t.thr_type == "0.5"]
    .set_index(key)[["acc", "f1_fire"]]
    .rename(columns={"acc": "acc_05", "f1_fire": "f1_05"})
)
M = M.join(
    t[t.thr_type == "val"]
    .set_index(key)[["recall_fire", "far", "thr"]]
    .rename(columns={"recall_fire": "recall_val", "far": "far_val", "thr": "thr_val"})
)
M = M.join(res.drop_duplicates(key).set_index(key)[["seconds"]]).reset_index()

COLS = ["roc_auc", "pr_auc", "far_r95", "acc_05", "f1_05", "recall_val", "far_val"]


def fmt(df, cols):
    g = df.groupby("model")[cols]
    mu, sd = g.mean(), g.std(ddof=1).fillna(0)
    return (mu.round(4).astype(str) + " ± " + sd.round(4).astype(str)).loc[models]


summary = fmt(M[M.exp == "Exp2"], COLS)
print("Test, Exp2 (mean ± sd antar seed)")
display(summary)
print("\nTest, Exp1 (mean ± sd antar seed)")
display(fmt(M[M.exp == "Exp1"], COLS))


## 16. Analisis: perbandingan berpasangan

In [ ]:
S_, L_ = "mnv3s", "mnv3l"
PAIRS = []
for b in (S_, L_):
    PAIRS += [
        (f"{b}_mlp vs {b}", f"{b}_mlp", b),
        (f"{b}_fuse vs {b}", f"{b}_fuse", b),
        (f"{b}_fmlp vs {b}", f"{b}_fmlp", b),
        (f"{b}_fuse3 vs {b}", f"{b}_fuse3", b),
        (f"{b}_fuse vs {b}_mlp", f"{b}_fuse", f"{b}_mlp"),
        (f"{b}_fmlp vs {b}_fuse", f"{b}_fmlp", f"{b}_fuse"),
        (f"{b}_fmlp vs {b}_mlp", f"{b}_fmlp", f"{b}_mlp"),
        (f"{b}_fuse3 vs {b}_fuse", f"{b}_fuse3", f"{b}_fuse"),
    ]
PAIRS += [
    ("mnv3l vs mnv3s", L_, S_),
    ("mnv3s_mlp vs mnv3l", "mnv3s_mlp", L_),
    ("mnv3s_fuse vs mnv3l", "mnv3s_fuse", L_),
    ("mnv3s_fmlp vs mnv3l", "mnv3s_fmlp", L_),
    ("mnv3s vs mobilenetv2", S_, "mobilenetv2"),
    ("mnv3l vs mobilenetv2", L_, "mobilenetv2"),
]
PAIRS = [p for p in PAIRS if p[1] in models and p[2] in models]
SIGN = {"roc_auc": 1, "pr_auc": 1, "far_r95": -1, "acc_05": 1}


def holm(p):
    p = np.asarray(p, float)
    adj = np.empty_like(p)
    mx = 0
    for r, i in enumerate(np.argsort(p)):
        mx = max(mx, (len(p) - r) * p[i])
        adj[i] = min(1, mx)
    return adj


def paired_table(exp="Exp2"):
    out = []
    for lab, new, ref in PAIRS:
        for met in SIGN:
            a = M[(M.model == new) & (M.exp == exp)].set_index("seed")[met]
            b = M[(M.model == ref) & (M.exp == exp)].set_index("seed")[met]
            d = (a - b).dropna()
            n = len(d)
            mu = d.mean()
            sd = d.std(ddof=1) if n > 1 else np.nan
            if n > 1 and sd > 0:
                p = stats.ttest_1samp(d, 0).pvalue
                h = stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n)
            else:
                p, h = (0.0 if (n > 1 and mu != 0) else 1.0), np.nan
            out.append(
                dict(
                    pair=lab,
                    metric=met,
                    n_seed=n,
                    delta_mean=mu,
                    delta_sd=sd,
                    ci_lo=mu - h,
                    ci_hi=mu + h,
                    wins=int((d * SIGN[met] > 0).sum()),
                    p_t=p,
                )
            )
    r = pd.DataFrame(out)
    r["p_holm"] = r.groupby("metric").p_t.transform(lambda s: holm(s.values))
    return r


PT = paired_table("Exp2")
for met in ("roc_auc", "far_r95", "pr_auc", "acc_05"):
    print(f"\nΔ {met} (Exp2, baru − acuan)")
    display(
        PT[PT.metric == met]
        .set_index("pair")[
            [
                "n_seed",
                "delta_mean",
                "delta_sd",
                "ci_lo",
                "ci_hi",
                "wins",
                "p_t",
                "p_holm",
            ]
        ]
        .round(4)
    )


def seedavg(model, exp):
    g = pr[(pr.split == "test") & (pr.exp == exp) & (pr.model == model)]
    sc = (
        g.pivot_table(index="path", columns="seed", values="p_fire")
        .mean(axis=1)
        .sort_index()
    )
    y = g.drop_duplicates("path").set_index("path").y_true.loc[sc.index]
    return sc.values, y.values


def boot_delta(new, ref, exp="Exp2", B=1000, seed=0):
    pn, y = seedavg(new, exp)
    po, y2 = seedavg(ref, exp)
    assert (y == y2).all()
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        out.append(
            [
                roc_auc_score(y[i], pn[i]) - roc_auc_score(y[i], po[i]),
                far_at_recall(y[i], pn[i], 0.95) - far_at_recall(y[i], po[i], 0.95),
            ]
        )
    return np.quantile(np.array(out), [0.025, 0.5, 0.975], axis=0)


br = []
for lab, new, ref in PAIRS:
    q = boot_delta(new, ref)
    br.append(
        dict(
            pair=lab,
            dAUC=q[1, 0],
            dAUC_lo=q[0, 0],
            dAUC_hi=q[2, 0],
            dFAR95=q[1, 1],
            dFAR95_lo=q[0, 1],
            dFAR95_hi=q[2, 1],
        )
    )
BR = pd.DataFrame(br).set_index("pair")
BR["AUC_sig"] = (BR.dAUC_lo > 0) | (BR.dAUC_hi < 0)
BR["FAR95_sig"] = (BR.dFAR95_lo > 0) | (BR.dFAR95_hi < 0)
print("\nBootstrap citra (1000 ulangan), skor rata-rata antar seed, Exp2")
display(BR.round(4))

ft = (
    (
        M[M.exp == "Exp2"].set_index(["model", "seed"])[["roc_auc", "far_r95"]]
        - M[M.exp == "Exp1"].set_index(["model", "seed"])[["roc_auc", "far_r95"]]
    )
    .groupby("model")
    .agg(["mean", "std"])
    .loc[models]
)
print("\nEfek fine-tuning (Exp2 − Exp1)")
display(ft.round(4))

## 17. Analisis: kurva

In [ ]:
s0 = common[0]
nc = 4
nr = int(np.ceil(len(models) / nc))
fig, ax = plt.subplots(nr, nc, figsize=(4.5 * nc, 3.2 * nr), squeeze=False)
for a in ax.ravel():
    a.axis("off")
for a, m in zip(ax.ravel(), models):
    a.axis("on")
    e = ep[(ep.model == m) & (ep.seed == s0)].reset_index(drop=True)
    n1 = int((e.exp == "Exp1").sum())
    x = np.arange(1, len(e) + 1)
    a.plot(x, e.loss, label="train")
    a.plot(x, e.val_loss, label="val")
    a.axvline(n1 + 0.5, ls="--", c="gray")
    a.set_title(f"{m} (seed {s0})", fontsize=8)
    a.grid(alpha=0.3)
ax[0, 0].legend(fontsize=7)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/reports/loss_curves.png", dpi=200)
plt.show()

MAIN = [
    m
    for m in (
        "mobilenetv2",
        "mnv3s",
        "mnv3s_mlp",
        "mnv3s_fuse",
        "mnv3l",
        "mnv3l_mlp",
        "mnv3l_fuse",
    )
    if m in models
]
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for m in MAIN:
    p, y = seedavg(m, "Exp2")
    fpr, tpr, _ = roc_curve(y, p)
    pc, rc, _ = precision_recall_curve(y, p)
    ax[0].plot(fpr, tpr, label=f"{m} {roc_auc_score(y, p):.3f}")
    ax[1].plot(rc, pc, label=f"{m} {average_precision_score(y, p):.3f}")
ax[0].axhline(0.95, ls=":", c="gray")
ax[0].set_title("ROC test, Exp2 (skor rata-rata antar seed)")
ax[1].set_title("PR test, Exp2 (skor rata-rata antar seed)")
for a in ax:
    a.legend(fontsize=7)
    a.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/reports/roc_pr.png", dpi=200)
plt.show()

## 18. Analisis: trade-off akurasi vs biaya

In [ ]:
REF = "mnv3l" if "mnv3l" in models else models[0]
cx = (
    cfg.groupby("model")[
        ["params_total", "params_head", "flops_g", "latency_ms_b1", "throughput_ips"]
    ]
    .mean()
    .loc[models]
)
tot = M.groupby(["model", "seed"]).seconds.sum().groupby("model").mean()
e2 = M[M.exp == "Exp2"].groupby("model")
tab = pd.DataFrame(
    {
        "params_M": cx.params_total / 1e6,
        "head_params_k": cx.params_head / 1e3,
        "GFLOPs": cx.flops_g,
        "latency_ms_b1": cx.latency_ms_b1,
        "img_per_s": cx.throughput_ips,
        "train_min": tot / 60,
        "roc_auc": e2.roc_auc.mean(),
        "roc_auc_sd": e2.roc_auc.std().fillna(0),
        "far_r95": e2.far_r95.mean(),
        "far_r95_sd": e2.far_r95.std().fillna(0),
    }
).loc[models]
for c in ("params_M", "GFLOPs", "img_per_s"):
    tab[c + "_x_ref"] = tab[c] / tab.loc[REF, c]
tab = tab.round(4)
display(tab)

col = {"mobilenetv2": "gray", "mnv3s": "tab:blue", "mnv3l": "tab:orange"}
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for a, y, lab in zip(
    ax,
    ("roc_auc", "far_r95"),
    ("ROC-AUC (test, Exp2)", "FAR @ recall 0.95 (test, Exp2)"),
):
    for m, r in tab.iterrows():
        a.errorbar(
            r.GFLOPs,
            r[y],
            yerr=r[y + "_sd"],
            fmt="o",
            c=col[m.split("_")[0]],
            capsize=3,
        )
        a.annotate(
            m, (r.GFLOPs, r[y]), fontsize=7, xytext=(3, 3), textcoords="offset points"
        )
    a.set_xscale("log")
    a.set_xlabel("GFLOPs (log)")
    a.set_ylabel(lab)
    a.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/reports/tradeoff.png", dpi=200)
plt.show()

## 19. Analisis: efek CLAHE

In [ ]:
os.makedirs(f"{OUT_DIR}/reports", exist_ok=True)
done_c = [
    d
    for d in sorted(glob.glob(f"{OUT_DIR}/*_img{IMG}_seed*"))
    if os.path.exists(f"{d}/DONE")
]
res_c = pd.concat([pd.read_csv(f"{d}/results.csv") for d in done_c], ignore_index=True)
res_c["thr_type"] = res_c.thr_type.astype(str)
pr_c = pd.concat([pd.read_csv(f"{d}/preds.csv") for d in done_c], ignore_index=True)
have = set(res_c.model)
CL = [m for m in CLAHE_MODELS if m in have and m + "_clahe" in have]
assert CL
CM = [x for m in CL for x in (m, m + "_clahe")]
cseeds = sorted(set.intersection(*[set(res_c[res_c.model == m].seed) for m in CM]))
assert cseeds

rows = []
sel = pr_c[(pr_c.split == "test") & pr_c.model.isin(CM) & pr_c.seed.isin(cseeds)]
for (m, s, e), g in sel.groupby(["model", "seed", "exp"]):
    y, p = g.y_true.values, g.p_fire.values
    rows.append(
        dict(
            model=m,
            seed=s,
            exp=e,
            roc_auc=roc_auc_score(y, p),
            pr_auc=average_precision_score(y, p),
            far_r95=far_at_recall(y, p, 0.95),
        )
    )
MC = pd.DataFrame(rows)
key = ["model", "seed", "exp"]
t = res_c[(res_c.split == "test") & res_c.model.isin(CM) & res_c.seed.isin(cseeds)]
MC = MC.merge(
    t[t.thr_type == "0.5"][key + ["acc", "f1_fire"]].rename(
        columns={"acc": "acc_05", "f1_fire": "f1_05"}
    ),
    on=key,
)
MC = MC.merge(
    t[t.thr_type == "val"][key + ["recall_fire", "far"]].rename(
        columns={"recall_fire": "recall_val", "far": "far_val"}
    ),
    on=key,
)

CC = ["roc_auc", "pr_auc", "far_r95", "acc_05", "f1_05", "recall_val", "far_val"]


def fmt_c(df):
    g = df.groupby("model")[CC]
    mu, sd = g.mean(), g.std(ddof=1).fillna(0)
    return (mu.round(4).astype(str) + " ± " + sd.round(4).astype(str)).loc[CM]


clahe_summary = fmt_c(MC[MC.exp == "Exp2"])
print("Test, Exp2: tanpa CLAHE vs dengan CLAHE (mean ± sd antar seed)")
display(clahe_summary)
print("\nTest, Exp1 (mean ± sd antar seed)")
display(fmt_c(MC[MC.exp == "Exp1"]))

SIGN_C = {"roc_auc": 1, "pr_auc": 1, "far_r95": -1, "acc_05": 1}
PAIRS_C = [(f"{m}_clahe vs {m}", m + "_clahe", m) for m in CL]
for b in ("mnv3s", "mnv3l"):
    if b in CL and b + "_fmlp" in CL:
        PAIRS_C.append(
            (f"{b}_fmlp_clahe vs {b}_clahe", b + "_fmlp_clahe", b + "_clahe")
        )
if "mnv3s_fmlp_clahe" in CM and "mnv3l_clahe" in CM:
    PAIRS_C.append(
        ("mnv3s_fmlp_clahe vs mnv3l_clahe", "mnv3s_fmlp_clahe", "mnv3l_clahe")
    )


def paired_c(exp="Exp2"):
    out = []
    for lab, new, ref in PAIRS_C:
        for met in SIGN_C:
            a = MC[(MC.model == new) & (MC.exp == exp)].set_index("seed")[met]
            b = MC[(MC.model == ref) & (MC.exp == exp)].set_index("seed")[met]
            d = (a - b).dropna()
            n = len(d)
            mu = d.mean()
            sd = d.std(ddof=1) if n > 1 else np.nan
            if n > 1 and sd > 0:
                p = stats.ttest_1samp(d, 0).pvalue
                h = stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n)
            else:
                p, h = (0.0 if (n > 1 and mu != 0) else 1.0), np.nan
            out.append(
                dict(
                    pair=lab,
                    metric=met,
                    n_seed=n,
                    delta_mean=mu,
                    delta_sd=sd,
                    ci_lo=mu - h,
                    ci_hi=mu + h,
                    wins=int((d * SIGN_C[met] > 0).sum()),
                    p_t=p,
                )
            )
    r = pd.DataFrame(out)
    r["p_holm"] = r.groupby("metric").p_t.transform(lambda s: holm(s.values))
    return r


PT_C = paired_c("Exp2")
for met in ("roc_auc", "far_r95", "pr_auc", "acc_05"):
    print(f"\nΔ {met} (Exp2, baru − acuan)")
    display(
        PT_C[PT_C.metric == met]
        .set_index("pair")[
            [
                "n_seed",
                "delta_mean",
                "delta_sd",
                "ci_lo",
                "ci_hi",
                "wins",
                "p_t",
                "p_holm",
            ]
        ]
        .round(4)
    )

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, met, lab in zip(
    ax,
    ("roc_auc", "far_r95"),
    ("Δ ROC-AUC (CLAHE − tanpa)", "Δ far_r95 (CLAHE − tanpa)"),
):
    for i, m in enumerate(CL):
        d = (
            MC[(MC.model == m + "_clahe") & (MC.exp == "Exp2")].set_index("seed")[met]
            - MC[(MC.model == m) & (MC.exp == "Exp2")].set_index("seed")[met]
        ).dropna()
        a.scatter([i] * len(d), d.values, c="gray", s=18, zorder=3)
        a.scatter(i, d.mean(), c="tab:red", marker="_", s=500, zorder=4)
    a.axhline(0, ls="--", c="k", lw=0.8)
    a.set_xticks(range(len(CL)))
    a.set_xticklabels(CL, rotation=20, fontsize=8)
    a.set_ylabel(lab)
    a.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/reports/clahe_effect.png", dpi=200)
plt.show()

clahe_summary.to_csv(f"{OUT_DIR}/reports/clahe_summary_exp2_mean_sd.csv")
MC.to_csv(f"{OUT_DIR}/reports/clahe_metrics_per_run.csv", index=False)
PT_C.to_csv(f"{OUT_DIR}/reports/clahe_paired_deltas.csv", index=False)

## 20. Simpan hasil

In [ ]:
summary.to_csv(f"{OUT_DIR}/reports/summary_exp2_mean_sd.csv")
M.to_csv(f"{OUT_DIR}/reports/metrics_per_run.csv", index=False)
PT.to_csv(f"{OUT_DIR}/reports/paired_seed_deltas.csv", index=False)
BR.to_csv(f"{OUT_DIR}/reports/bootstrap_seedavg.csv")
ft.to_csv(f"{OUT_DIR}/reports/finetune_effect.csv")
tab.to_csv(f"{OUT_DIR}/reports/tradeoff.csv")
